In [118]:
import math
import time
import numpy as np

In [119]:
with open(r"C:\Users\giona\Documents\GitHub\5g-networks\notebooks\dataset\entropy.txt", "r", encoding="utf-8") as file:
    text = file.read()
n = 10000
print(f"Primi {n} caratteri del testo:\n")
print(text[:n]) 
# print(text)
print("\nLUNGHEZZA TESTO:\t", len(text), "char")
print("DIMENSIONE IN BIT:\t", len(text) * 8, "bit")

Primi 10000 caratteri del testo:

A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.
   The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspec

In [126]:
def lz78_encode(data: bytes):
    dizionario = {}
    coppie = []
    corrente = b""
    for b in data:
        c = bytes([b])
        if corrente + c in dizionario:
            corrente += c
        else:
            coppie.append((dizionario.get(corrente, 0), b))
            dizionario[corrente + c] = len(dizionario) + 1
            corrente = b""
    if corrente:
        coppie.append((dizionario.get(corrente[:-1], 0), corrente[-1]))
    return coppie


def lz78_decode(coppie) -> bytes:
    dizionario = [b""]
    out = bytearray()
    for indice, simbolo in coppie:
        s = dizionario[indice] + bytes([simbolo])
        out += s
        dizionario.append(s)
    return bytes(out)


def coppie_to_bits(coppie) -> str:
    bits = format(len(coppie), "032b")           # header: numero di coppie
    for i, (indice, simbolo) in enumerate(coppie):
        w = i.bit_length()                       # larghezza indice
        if w:
            bits += format(indice, f"0{w}b")
        bits += format(simbolo, "08b")
    return bits


def bits_to_coppie(bits: str):
    n = int(bits[:32], 2)
    pos = 32
    coppie = []
    for i in range(n):
        w = i.bit_length()
        indice = int(bits[pos:pos + w], 2) if w else 0
        pos += w
        simbolo = int(bits[pos:pos + 8], 2)
        pos += 8
        coppie.append((indice, simbolo))
    return coppie


In [129]:
print(len(coppie_to_bits(lz78_encode(text.encode("utf-8")))))

10601


In [128]:
# --- TEST LZ78 ---
print("="*40)
print("TEST COMPRESSIONE LZ78")
print("="*40)

start_time = time.time()
# Compressione
coppie_lz78 = lz78_encode(text.encode("utf-8"))
bit_lz78 = coppie_to_bits(coppie_lz78)
lz78_comp_time = time.time() - start_time

start_time = time.time()
# Decompressione (usiamo le coppie direttamente)
testo_decompresso_lz78 = lz78_decode(bits_to_coppie(bit_lz78)).decode("utf-8")
lz78_decomp_time = time.time() - start_time

# Statistiche LZ78
dim_originale_bit = len(text) * 8
dim_lz78_bit = len(bit_lz78)
rapporto_lz78 = dim_originale_bit / dim_lz78_bit
perc_risparmio_lz78 = (1 - (dim_lz78_bit / dim_originale_bit)) * 100

print(f"Testo originale recuperato con successo? {'SI' if text == testo_decompresso_lz78 else 'NO'}")
print(f"Dimensione originale: {dim_originale_bit} bit")
print(f"Dimensione compresso: {dim_lz78_bit} bit")
print(f"Rapporto di compressione: {rapporto_lz78:.2f}x")
print(f"Spazio risparmiato: {perc_risparmio_lz78:.2f}%")
print(f"Tempo di compressione: {lz78_comp_time:.4f} s")
print(f"Tempo di decompressione: {lz78_decomp_time:.4f} s\n")

TEST COMPRESSIONE LZ78
Testo originale recuperato con successo? SI
Dimensione originale: 14736 bit
Dimensione compresso: 10601 bit
Rapporto di compressione: 1.39x
Spazio risparmiato: 28.06%
Tempo di compressione: 0.0030 s
Tempo di decompressione: 0.0000 s

